# LeetCode 2485 — Find the Pivot Integer bằng Python

Link đề: https://leetcode.com/problems/find-the-pivot-integer/ · Độ khó: **Easy** · Chủ đề: Math, Prefix Sum, Binary Search, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def pivotInteger(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm số nguyên `x` trong `1..n` sao cho **tổng `1 + 2 + ... + x` bằng tổng `x + (x+1) + ... + n`** (cả hai tổng đều **bao gồm `x`**), hoặc trả về `-1`.
2. Dùng công thức tổng cấp số cộng (Gauss): `1 + 2 + ... + m = m(m+1)/2`, và `sum(range(...))` của Python.
3. Biến điều kiện thành phương trình rất gọn: `x^2 = n(n+1)/2`, từ đó giải bằng **căn bậc hai nguyên** `math.isqrt` — một lời giải `O(1)`.
4. Viết 5 cách: brute force `O(n^2)`, tổng tiền tố chạy `O(n)`, công thức + `isqrt` `O(1)`, tìm kiếm nhị phân `O(log n)`, và hai con trỏ; đối chiếu bằng test với mọi `n` từ 1 đến 1000.
5. Tránh các lỗi: dùng `sqrt` làm tròn mà không kiểm tra lại, tính tổng bên phải **không bao gồm** `x`, bỏ sót `x = n`, chia `/` ra số thực.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Tìm **số nguyên trục** (pivot integer) `x` sao cho:

> tổng các số từ `1` đến `x` (bao gồm `x`) **bằng** tổng các số từ `x` đến `n` (bao gồm `x`).

Trả về `x`. Nếu không có `x` nào, trả về `-1`. Đề đảm bảo với mỗi `n` có **nhiều nhất một** số trục.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `8` | 6 | `1+2+3+4+5+6 = 21` và `6+7+8 = 21` |
| 2 | `1` | 1 | `x = 1`: `1 = 1` |
| 3 | `4` | -1 | không có `x` nào thoả |

Constraints: `1 <= n <= 1000`.

In [1]:
n = 8
for x in range(1, n + 1):
    left = sum(range(1, x + 1))       # 1 + ... + x
    right = sum(range(x, n + 1))      # x + ... + n
    print(f"x={x}: trái={left}, phải={right}", "<- pivot" if left == right else "")

x=1: trái=1, phải=36 
x=2: trái=3, phải=35 
x=3: trái=6, phải=33 
x=4: trái=10, phải=30 
x=5: trái=15, phải=26 
x=6: trái=21, phải=21 <- pivot
x=7: trái=28, phải=15 
x=8: trái=36, phải=8 


## 3. Kiến thức nền

### 3.1. `range` và `sum(range(a, b))`

`range(a, b)` sinh các số `a, a+1, ..., b-1` (**không gồm `b`**). Muốn tính `1 + ... + x` ta viết `sum(range(1, x + 1))`; muốn `x + ... + n` ta viết `sum(range(x, n + 1))`. Rất dễ quên `+ 1` ở cận trên.

In [2]:
print(list(range(1, 5)))
print(sum(range(1, 5)))          # 1 + 2 + 3 + 4
print(sum(range(3, 9)))          # 3 + 4 + ... + 8

[1, 2, 3, 4]
10
33


### 3.2. Công thức tổng cấp số cộng (Gauss)

`1 + 2 + ... + m = m(m + 1) / 2`. Ghép cặp: `(1 + m) + (2 + (m-1)) + ...` mỗi cặp bằng `m + 1`, có `m/2` cặp. Trong Python dùng `//` (chia nguyên) cho kết quả số nguyên: `m * (m + 1) // 2` (tích hai số liên tiếp luôn chẵn nên chia hết).

Tổng đoạn `x..n` = tổng `1..n` trừ tổng `1..x-1`.

In [3]:
def tri(m):
    return m * (m + 1) // 2

print(tri(4), tri(8), tri(0))
print(tri(8) - tri(5))           # 6 + 7 + 8
print(sum(range(6, 9)))

10 36 0
21
21


### 3.3. Tổng tiền tố chạy (running prefix sum)

Khi duyệt `x = 1, 2, 3, ...`, tổng trái là `left += x`; tổng phải là `total - (left - x)` (tổng cả dãy trừ phần đã nằm hẳn bên trái `x`). Nhờ vậy mỗi `x` chỉ tốn `O(1)`.

In [4]:
n = 8
total = n * (n + 1) // 2
left = 0
for x in range(1, n + 1):
    left += x
    right = total - left + x
    print(f"x={x}: left={left}, right={right}")

x=1: left=1, right=36
x=2: left=3, right=35
x=3: left=6, right=33
x=4: left=10, right=30
x=5: left=15, right=26
x=6: left=21, right=21
x=7: left=28, right=15
x=8: left=36, right=8


### 3.4. Căn bậc hai nguyên `math.isqrt`

`math.isqrt(m)` trả về `floor(sqrt(m))` như số nguyên, **chính xác** với mọi số nguyên (khác `math.sqrt` trả số thực có thể sai số). Một số `m` là số chính phương khi và chỉ khi `isqrt(m) ** 2 == m`.

In [5]:
import math
for m in [36, 10, 1, 0, 49, 50]:
    r = math.isqrt(m)
    print(f"m={m}: isqrt={r}, r*r={r * r}, chính phương? {r * r == m}")

m=36: isqrt=6, r*r=36, chính phương? True
m=10: isqrt=3, r*r=9, chính phương? False
m=1: isqrt=1, r*r=1, chính phương? True
m=0: isqrt=0, r*r=0, chính phương? True
m=49: isqrt=7, r*r=49, chính phương? True
m=50: isqrt=7, r*r=49, chính phương? False


### 3.5. Tìm kiếm nhị phân trên hàm đơn điệu

Nếu một hàm `f(x)` **tăng dần** theo `x`, ta tìm `x` có `f(x) == 0` bằng cách giữ khoảng `[lo, hi]`: xét `mid`; nếu `f(mid) < 0` thì đáp án (nếu có) nằm bên phải (`lo = mid + 1`), nếu `> 0` thì bên trái (`hi = mid - 1`), nếu `== 0` thì xong. Mỗi bước giảm một nửa khoảng nên chỉ `O(log n)` bước.

In [6]:
def find(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"lo={lo} hi={hi} mid={mid} arr[mid]={arr[mid]}")
        if arr[mid] == target:
            return mid
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print(find([1, 3, 5, 7, 9, 11], 9))

lo=0 hi=5 mid=2 arr[mid]=5
lo=3 hi=5 mid=4 arr[mid]=9
4


### 3.6. Hai con trỏ từ hai đầu với tổng chạy

Giữ `l` (bắt đầu 1) và `r` (bắt đầu `n`), cùng hai tổng `lsum` (tổng của `1..l`) và `rsum` (tổng của `r..n`). Nếu `lsum < rsum` thì mở rộng bên trái (`l += 1`, `lsum += l`), ngược lại thu hẹp bên phải (`r -= 1`, `rsum += r`). Hai con trỏ gặp nhau tại một số; nếu lúc đó hai tổng bằng nhau thì số đó là pivot.

In [7]:
n = 8
l, r = 1, n
lsum, rsum = l, r
while l < r:
    if lsum < rsum:
        l += 1
        lsum += l
    else:
        r -= 1
        rsum += r
    print(f"l={l} r={r} lsum={lsum} rsum={rsum}")

l=2 r=8 lsum=3 rsum=8
l=3 r=8 lsum=6 rsum=8
l=4 r=8 lsum=10 rsum=8
l=4 r=7 lsum=10 rsum=15
l=5 r=7 lsum=15 rsum=15
l=5 r=6 lsum=15 rsum=21
l=6 r=6 lsum=21 rsum=21


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thử từng `x` và so hai tổng — cách làm tự nhiên nhất. Tính tay với `n = 8` (tổng toàn dãy `1..8 = 36`):

```
x  | tổng trái 1..x | tổng phải x..8
1  |  1             | 36
2  |  3             | 35
3  |  6             | 33
4  | 10             | 30
5  | 15             | 26
6  | 21             | 21   <- bằng nhau
7  | 28             | 15
```

Quan sát: tổng trái **tăng**, tổng phải **giảm** khi `x` tăng. Do đó hiệu `trái - phải` **tăng ngặt** theo `x`; nó cắt số `0` tối đa một lần (đúng với lời đề "nhiều nhất một số trục"). Điều này gợi ý hai hướng nhanh hơn: tìm nhị phân, hoặc giải thẳng phương trình.

## 5. Cách 1 — Brute Force / baseline

Với mỗi `x` từ `1` đến `n`, tính hai tổng **từ đầu** bằng `sum(range(...))` rồi so sánh.

Nút thắt: mỗi `x` tốn `O(n)` để cộng, có `n` giá trị `x` → `O(n^2)`. Mỗi lần ta cộng lại các số đã cộng ở lần trước.

In [8]:
class SolutionBrute:
    def pivotInteger(self, n):
        for x in range(1, n + 1):
            if sum(range(1, x + 1)) == sum(range(x, n + 1)):
                return x
        return -1

print(SolutionBrute().pivotInteger(8))
print(SolutionBrute().pivotInteger(1))
print(SolutionBrute().pivotInteger(4))

6
1
-1


**Độ phức tạp:** Time `O(n^2)`, Space `O(1)`. Với `n <= 1000` chỉ ~500.000 phép cộng nên vẫn nhanh, nhưng chưa đẹp.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `T = n(n+1)/2` là tổng `1..n`. Với `x` bất kỳ:

- Tổng trái `L(x) = x(x+1)/2`.
- Tổng phải `R(x) = T - L(x-1) = T - x(x-1)/2`.

`L(x) = R(x)` ⇔ `x(x+1)/2 + x(x-1)/2 = T` ⇔ `x^2 = T`.

> **Pivot tồn tại khi và chỉ khi `T = n(n+1)/2` là số chính phương, và khi đó `x = √T`.**

(Cộng hai tổng, phần `x` được tính hai lần: `L + R = T + x`; mà `L = R` nên `2L = T + x`, tức `x(x+1) = T + x`, suy ra `x^2 = T`.)

**Quy tắc "nếu… thì…":**

- Nếu `isqrt(T) ** 2 == T` → trả `isqrt(T)`.
- Ngược lại → `-1`.
- Nếu muốn không dùng công thức → hiệu `L(x) - R(x) = x^2 - T` tăng ngặt, nên có thể tìm nhị phân hoặc hai con trỏ.

Invariant (hai con trỏ): sau mỗi bước, `lsum` là tổng `1..l`, `rsum` là tổng `r..n`, và ta luôn mở rộng phía có tổng nhỏ hơn.

In [9]:
for n in [8, 1, 4, 49, 288, 10]:
    T = n * (n + 1) // 2
    r = math.isqrt(T)
    print(f"n={n}: T={T}, isqrt={r}, chính phương? {r * r == T} -> đáp án {r if r * r == T else -1}")

n=8: T=36, isqrt=6, chính phương? True -> đáp án 6
n=1: T=1, isqrt=1, chính phương? True -> đáp án 1
n=4: T=10, isqrt=3, chính phương? False -> đáp án -1
n=49: T=1225, isqrt=35, chính phương? True -> đáp án 35
n=288: T=41616, isqrt=204, chính phương? True -> đáp án 204
n=10: T=55, isqrt=7, chính phương? False -> đáp án -1


## 7. Các cách tiếp theo

Ta có 5 cách: (1) brute force `O(n^2)`; (2) tổng chạy `O(n)`; (3) công thức `x^2 = T` + `isqrt` `O(1)`; (4) tìm kiếm nhị phân `O(log n)`; (5) hai con trỏ `O(n)`.

### Cách 2 — Tổng chạy (prefix sum) một lượt

Tính `T` một lần. Khi duyệt `x`, cộng dồn `left += x`, còn tổng phải suy ra bằng `T - left + x` (vì `left` đã gồm `x`, mà phải thì cũng gồm `x`, nên phải trừ đi phần `1..x` rồi cộng lại `x`).

In [10]:
class SolutionPrefix:
    def pivotInteger(self, n):
        total = n * (n + 1) // 2
        left = 0
        for x in range(1, n + 1):
            left += x
            right = total - left + x
            if left == right:
                return x
        return -1

print(SolutionPrefix().pivotInteger(8))
print(SolutionPrefix().pivotInteger(1))
print(SolutionPrefix().pivotInteger(4))

6
1
-1


Độ phức tạp Cách 2: Time `O(n)`, Space `O(1)`.

### Cách 3 — Công thức `x^2 = n(n+1)/2` + `isqrt` (toán học)

Tính `T`, lấy `r = isqrt(T)`, kiểm tra `r * r == T`. Dùng `isqrt` (số nguyên) thay vì `sqrt` (số thực) để tránh sai số làm tròn.

In [11]:
class SolutionMath:
    def pivotInteger(self, n):
        total = n * (n + 1) // 2
        root = math.isqrt(total)
        return root if root * root == total else -1

print(SolutionMath().pivotInteger(8))
print(SolutionMath().pivotInteger(1))
print(SolutionMath().pivotInteger(4))

6
1
-1


Độ phức tạp Cách 3: Time `O(1)` (một `isqrt` trên số nhỏ), Space `O(1)`.

### Cách 4 — Tìm kiếm nhị phân

Vì `g(x) = L(x) - R(x) = x^2 - T` tăng ngặt trên `x >= 1`, ta tìm nhị phân trong `[1, n]` điểm `g(x) = 0`. Dùng công thức đóng cho `L` và `R` để mỗi lần so sánh tốn `O(1)`.

In [12]:
class SolutionBinary:
    def pivotInteger(self, n):
        total = n * (n + 1) // 2
        lo, hi = 1, n
        while lo <= hi:
            mid = (lo + hi) // 2
            left = mid * (mid + 1) // 2
            right = total - mid * (mid - 1) // 2
            if left == right:
                return mid
            if left < right:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1

print(SolutionBinary().pivotInteger(8))
print(SolutionBinary().pivotInteger(1))
print(SolutionBinary().pivotInteger(4))

6
1
-1


Độ phức tạp Cách 4: Time `O(log n)`, Space `O(1)`.

### Cách 5 — Hai con trỏ với tổng chạy

Như mục 3.6: đẩy con trỏ phía có tổng nhỏ hơn. Khi `l == r` kiểm tra hai tổng. Không cần công thức Gauss hay căn bậc hai.

In [13]:
class SolutionTwoPointers:
    def pivotInteger(self, n):
        l, r = 1, n
        lsum, rsum = l, r
        while l < r:
            if lsum < rsum:
                l += 1
                lsum += l
            else:
                r -= 1
                rsum += r
        return l if lsum == rsum else -1

print(SolutionTwoPointers().pivotInteger(8))
print(SolutionTwoPointers().pivotInteger(1))
print(SolutionTwoPointers().pivotInteger(4))

6
1
-1


Độ phức tạp Cách 5: Time `O(n)` (mỗi bước con trỏ di chuyển một ô, tổng cộng `n - 1` bước), Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 (nhị phân) với `n = 8`, `T = 36`:

| vòng | `lo` | `hi` | `mid` | `left` | `right` | quyết định |
|---|---|---|---|---|---|---|
| 1 | 1 | 8 | 4 | 10 | 30 | `left < right` → `lo = 5` |
| 2 | 5 | 8 | 6 | 21 | 21 | bằng nhau → trả `6` |

Với `n = 4` (`T = 10`) vòng đầu `mid = 2` (`left=3`, `right=9`) → `lo = 3`; `mid = 3` (`left=6`, `right=7`) → `lo = 4`; `mid = 4` (`left=10`, `right=4`) → `hi = 3`; `lo > hi` → trả `-1`.

In [14]:
def trace_binary(n):
    total = n * (n + 1) // 2
    print(f"n={n}, T={total}")
    lo, hi = 1, n
    step = 0
    while lo <= hi:
        step += 1
        mid = (lo + hi) // 2
        left = mid * (mid + 1) // 2
        right = total - mid * (mid - 1) // 2
        print(f"vòng {step}: lo={lo} hi={hi} mid={mid} left={left} right={right}")
        if left == right:
            print("-> tìm thấy", mid)
            return mid
        if left < right:
            lo = mid + 1
        else:
            hi = mid - 1
    print("-> không có pivot")
    return -1

trace_binary(8)
trace_binary(4)
trace_binary(49)

n=8, T=36
vòng 1: lo=1 hi=8 mid=4 left=10 right=30
vòng 2: lo=5 hi=8 mid=6 left=21 right=21
-> tìm thấy 6
n=4, T=10
vòng 1: lo=1 hi=4 mid=2 left=3 right=9
vòng 2: lo=3 hi=4 mid=3 left=6 right=7
vòng 3: lo=4 hi=4 mid=4 left=10 right=4
-> không có pivot
n=49, T=1225
vòng 1: lo=1 hi=49 mid=25 left=325 right=925
vòng 2: lo=26 hi=49 mid=37 left=703 right=559
vòng 3: lo=26 hi=36 mid=31 left=496 right=760
vòng 4: lo=32 hi=36 mid=34 left=595 right=664
vòng 5: lo=35 hi=36 mid=35 left=630 right=630
-> tìm thấy 35


35

## 9. Code nộp LeetCode

In [15]:
import math

class Solution:
    def pivotInteger(self, n: int) -> int:
        total = n * (n + 1) // 2
        root = math.isqrt(total)
        return root if root * root == total else -1

print(Solution().pivotInteger(8), Solution().pivotInteger(1), Solution().pivotInteger(4))

6 1 -1


## 10. Vì sao thuật toán đúng

- **Phương trình:** `L(x) = x(x+1)/2` và `R(x) = T - x(x-1)/2` (tổng `x..n` bằng tổng `1..n` trừ tổng `1..x-1`). `L = R` ⇒ `x(x+1)/2 + x(x-1)/2 = T` ⇒ `x^2 = T`. Phép biến đổi đảo ngược được nên **chỉ cần và đủ**.
- **Điều kiện `1 <= x <= n`:** nếu `x^2 = n(n+1)/2` thì `x^2 <= n^2`, nên `x <= n`; và `x >= 1` vì `T >= 1`. Do đó nghiệm của phương trình tự động nằm trong miền hợp lệ.
- **Tính duy nhất:** `x ≥ 1` và `x^2 = T` có tối đa một nghiệm nguyên dương `x = √T`.
- **`isqrt` chính xác:** `isqrt(T)` là số nguyên lớn nhất có bình phương `<= T`, nên `isqrt(T)^2 == T` ⇔ `T` chính phương. Không có sai số làm tròn như số thực.
- **Nhị phân:** `g(x) = x^2 - T` tăng ngặt với `x >= 1`, nên nếu `g(mid) < 0` mọi `x <= mid` đều có `g < 0`, loại được nửa trái; tương tự khi `g(mid) > 0`.
- **Hai con trỏ:** khi `l == r = x`, `lsum` là tổng `1..x` và `rsum` là tổng `x..n`. Quy tắc luôn đẩy bên nhỏ hơn bảo đảm `l` và `r` gặp nhau tại điểm mà hai tổng cân bằng nhất (và xét bằng nhau nếu có).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số đã cho.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force (tính lại tổng) | `O(n^2)` | `O(1)` |
| 2. Tổng chạy | `O(n)` | `O(1)` |
| 3. Công thức + `isqrt` | `O(1)` | `O(1)` |
| 4. Tìm kiếm nhị phân | `O(log n)` | `O(1)` |
| 5. Hai con trỏ | `O(n)` | `O(1)` |

Trong thực tế với `n <= 1000` mọi cách đều rất nhanh. Cách 3 là gọn nhất; cách 2 và 5 không cần toán nên dễ nghĩ ra trong phỏng vấn.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1`, `n = 1000`, các `n` có pivot: `1, 8, 49, 288`).

In [16]:
impls = [SolutionBrute(), SolutionPrefix(), SolutionMath(), SolutionBinary(), SolutionTwoPointers(), Solution()]
tests = [
    (8, 6),
    (1, 1),
    (4, -1),
    (2, -1),
    (3, -1),
    (49, 35),             # 1..35 = 630 = 35..49
    (288, 204),           # n(n+1)/2 = 41616 = 204^2
    (1000, -1),           # biên trên
    (999, -1),
    (6, -1),
    (7, -1),
    (9, -1),
    (50, -1),             # ngay sau 49 (bắt lỗi round(sqrt) không kiểm tra)
]
for n, exp in tests:
    assert 1 <= n <= 1000
    for im in impls:
        assert im.pivotInteger(n) == exp, (type(im).__name__, n, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: **vét cạn** toàn bộ miền `n = 1..1000`, đối chiếu mọi cách với brute force (không cần sinh ngẫu nhiên vì miền nhỏ), rồi thêm kiểm tra ngẫu nhiên có seed.

In [17]:
import random

pivots = {}
for n in range(1, 1001):
    exp = SolutionBrute().pivotInteger(n)
    for im in impls[1:]:
        assert im.pivotInteger(n) == exp, (type(im).__name__, n, exp)
    if exp != -1:
        pivots[n] = exp
print("vét cạn 1..1000 OK; các n có pivot:", pivots)

random.seed(2485)
for _ in range(300):
    n = random.randint(1, 1000)
    exp = SolutionBrute().pivotInteger(n)
    assert all(im.pivotInteger(n) == exp for im in impls)
print("300 test ngẫu nhiên OK")

vét cạn 1..1000 OK; các n có pivot: {1: 1, 8: 6, 49: 35, 288: 204}


300 test ngẫu nhiên OK


Test 3: tính chất — nếu có pivot `x` thì hai tổng bằng nhau thật sự, `1 <= x <= n`, và `x * x == n(n+1)/2`.

In [18]:
for n in range(1, 1001):
    x = Solution().pivotInteger(n)
    if x != -1:
        assert 1 <= x <= n
        assert sum(range(1, x + 1)) == sum(range(x, n + 1))
        assert x * x == n * (n + 1) // 2
assert list(pivots) == [1, 8, 49, 288]
print("tính chất của pivot: OK")

tính chất của pivot: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `sqrt` làm tròn mà không kiểm tra lại

In [19]:
def wrong_round_sqrt(n):
    total = n * (n + 1) // 2
    return round(math.sqrt(total))        # SAI: không kiểm tra root * root == total

n = 4
print("sai :", wrong_round_sqrt(n))
print("đúng:", Solution().pivotInteger(n))
assert wrong_round_sqrt(n) != Solution().pivotInteger(n)

sai : 3
đúng: -1


Vì sao sai: `T = 10` không phải số chính phương nhưng `round(√10) = 3` vẫn ra một số. Điều kiện cần là **`x^2 = T` đúng tuyệt đối**; phải kiểm tra `root * root == total` rồi mới trả `root`, ngược lại trả `-1`.

### Lỗi 2: tổng bên phải không bao gồm `x`

In [20]:
def wrong_exclude_x(n):
    for x in range(1, n + 1):
        if sum(range(1, x + 1)) == sum(range(x + 1, n + 1)):   # SAI: bên phải bắt đầu từ x + 1
            return x
    return -1

n = 8
print("sai :", wrong_exclude_x(n))
print("đúng:", Solution().pivotInteger(n))
assert wrong_exclude_x(n) != Solution().pivotInteger(n)

sai : -1
đúng: 6


Vì sao sai: đề nói **cả hai** tổng đều gồm `x` (`1..x` và `x..n`). Nếu bỏ `x` khỏi một bên thì với `n = 8` không tồn tại `x` nào (vì `1..x` luôn khác `x+1..8` khi chỉ xét `x` nguyên) nên ra `-1` thay vì `6`. **Sửa:** `range(x, n + 1)`.

### Lỗi 3: cận trên của `range` làm mất `x = n`

In [21]:
def wrong_range(n):
    for x in range(1, n):                  # SAI: thiếu x = n, phải là range(1, n + 1)
        if sum(range(1, x + 1)) == sum(range(x, n + 1)):
            return x
    return -1

n = 1
print("sai :", wrong_range(n))
print("đúng:", Solution().pivotInteger(n))
assert wrong_range(n) != Solution().pivotInteger(n)

sai : -1
đúng: 1


Vì sao sai: `range(1, 1)` rỗng nên với `n = 1` không xét `x = 1` → trả `-1` thay vì `1`. `range(a, b)` **không gồm `b`**; cần `range(1, n + 1)`.

### Lỗi 4 (bonus): dùng `/` thay `//`

`total = n * (n + 1) / 2` ra số **thực** (`36.0`); `math.isqrt(36.0)` báo lỗi vì cần số nguyên. Luôn dùng `//` khi muốn chia nguyên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng một đoạn bằng tổng đoạn kia" + dãy `1..n` → thử viết bằng công thức Gauss; thường thu về phương trình đại số rất gọn (ở đây `x^2 = n(n+1)/2`).
- Cần căn bậc hai **nguyên chính xác** → `math.isqrt`, không dùng `sqrt` + so sánh số thực.
- Hàm `f(x)` đơn điệu theo `x` → nghĩ ngay tới **tìm kiếm nhị phân** hoặc hai con trỏ.
- `n <= 1000` → brute force `O(n^2)` vẫn qua; hãy viết trước để làm oracle, rồi tối ưu.
- Khi miền đầu vào nhỏ (như `1..1000`), có thể **vét cạn toàn bộ miền** trong test để chắc chắn.
- Nhớ: `range(a, b)` không gồm `b`; tổng `x..n` bằng tổng `1..n` trừ tổng `1..x-1`.

## 15. Biến thể của bài

1. Bên trái **không** gồm `x`: tìm `x` sao cho `1..x-1` bằng `x+1..n` (liên quan tới "Find Pivot Index" trên mảng).
2. Tìm pivot cho dãy `a, a+d, a+2d, ...` bất kỳ (cấp số cộng tổng quát).
3. Cho `n` lớn tới `10^18`: bắt buộc dùng công thức + `isqrt`, và bàn về tràn số ở ngôn ngữ khác.
4. Liệt kê **tất cả** `n <= N` có pivot (các số "tam giác vuông" kiểu `1, 8, 49, 288, 1681, ...` — nghiệm phương trình Pell).
5. Trên **mảng** `nums`, tìm chỉ số pivot mà tổng bên trái bằng tổng bên phải (LeetCode 724 — Find Pivot Index).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | tính lại hai tổng cho mỗi `x` | `O(n^2)` | `O(1)` | hiểu đề, oracle |
| 2. Tổng chạy | cộng dồn, suy tổng phải từ `T` | `O(n)` | `O(1)` | đơn giản, không cần toán |
| 3. Công thức + `isqrt` | `x^2 = n(n+1)/2` | `O(1)` | `O(1)` | **gọn nhất**, `n` rất lớn |
| 4. Nhị phân | `x^2 - T` tăng ngặt | `O(log n)` | `O(1)` | khi quen tìm kiếm trên đáp án |
| 5. Hai con trỏ | đẩy bên có tổng nhỏ hơn | `O(n)` | `O(1)` | không cần công thức Gauss |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `pivotInteger(49)` bằng bao nhiêu? Kiểm tra bằng cách tính tổng `1..35` và `35..49`.

**Bài 2 (tính tay):** với `n = 6`, tính `T = 21` rồi cho biết có phải số chính phương không; liệt kê `L(x)` và `R(x)` cho `x = 1..6`.

**Bài 3 (điền TODO):** hoàn thành `is_perfect_square(m)` bằng `math.isqrt`.

**Bài 4 (tự viết lại):** viết `pivots_up_to(N)` trả về danh sách `[(n, x), ...]` mọi `n <= N` có pivot.

**Bài 5 (mini challenge):** viết `pivot_index(nums)` (LeetCode 724): chỉ số `i` mà tổng bên trái (không gồm `i`) bằng tổng bên phải (không gồm `i`), trả chỉ số **nhỏ nhất** hoặc `-1`.

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().pivotInteger(49))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def is_perfect_square(m):
    root = 0
    # TODO: root = math.isqrt(m); trả về root * root == m
    return False

35


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** hai tổng đều gồm `x`; tổng cả dãy `1..n` là `n(n+1)/2`.

**Mức 2 (vừa):** Bài 3: `root = math.isqrt(m)` rồi so `root * root` với `m`. Bài 4: lặp `n` từ 1 đến `N`, gọi `Solution().pivotInteger(n)` và giữ lại kết quả khác `-1`. Bài 5: dùng tổng toàn mảng `S` và tổng trái chạy; điều kiện `left == S - left - nums[i]`.

**Mức 3 (gần đáp án):** Bài 1: `35 * 36 / 2 = 630` và `49 * 50 / 2 - 34 * 35 / 2 = 1225 - 595 = 630` → `35`. Bài 2: `T = 21` không phải số chính phương (`4^2 = 16`, `5^2 = 25`) → `-1`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1: 1..35 = 630 và 35..49 = 630 -> 35
assert Solution().pivotInteger(49) == 35
assert sum(range(1, 36)) == sum(range(35, 50)) == 630

# Bài 2: n = 6, T = 21 không chính phương -> -1
assert 6 * 7 // 2 == 21 and math.isqrt(21) ** 2 != 21
assert Solution().pivotInteger(6) == -1
assert [(x, sum(range(1, x + 1)), sum(range(x, 7))) for x in (1, 6)] == [(1, 1, 21), (6, 21, 6)]

# Bài 3
def is_perfect_square(m):
    root = math.isqrt(m)
    return root * root == m
assert is_perfect_square(36) and is_perfect_square(1) and is_perfect_square(0)
assert not is_perfect_square(10) and not is_perfect_square(50)

# Bài 4
def pivots_up_to(N):
    result = []
    for n in range(1, N + 1):
        x = Solution().pivotInteger(n)
        if x != -1:
            result.append((n, x))
    return result
assert pivots_up_to(1000) == [(1, 1), (8, 6), (49, 35), (288, 204)]
assert pivots_up_to(7) == [(1, 1)]

# Bài 5
def pivot_index(nums):
    S = sum(nums)
    left = 0
    for i, v in enumerate(nums):
        if left == S - left - v:
            return i
        left += v
    return -1
assert pivot_index([1, 7, 3, 6, 5, 6]) == 3
assert pivot_index([1, 2, 3]) == -1
assert pivot_index([2, 1, -1]) == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "dãy `1..n` chia hai phần có tổng bằng nhau":**

```python
import math

total = n * (n + 1) // 2          # tổng 1..n, dùng // để ra số nguyên
root = math.isqrt(total)          # căn nguyên chính xác
return root if root * root == total else -1
```

**Checklist trước khi submit:**

- [ ] Hai tổng đều **gồm** `x` (`1..x` và `x..n`)?
- [ ] Dùng `//` (chia nguyên), không dùng `/`?
- [ ] Dùng `math.isqrt` và **kiểm tra** `root * root == total` trước khi trả đáp án?
- [ ] Nếu duyệt: `range(1, n + 1)` để không mất `x = n` (đặc biệt `n = 1`)?
- [ ] Đã thử `n = 1`, `n = 4` (không có), `n = 8`, `n = 49`?
- [ ] Đã đối chiếu các cách với brute force trên **toàn bộ** `n = 1..1000`?
- [ ] Chữ ký đúng `pivotInteger(self, n: int) -> int`?